# Liquidity-Aware CVaR Hedging with Order-Book Learning
## Phase I — literature, displayed-depth acquisition, reconstruction and cleaning

**Draft research report; not yet a completed empirical submission.** This notebook
runs without IBKR using a labeled **synthetic protocol fixture**. A real-data mode
reads one privately exported, stopped depth session. It does not train a predictor,
solve a CVaR hedge or implement LSV. Those are subsequent project stages.

The shared platform is pre-existing; this incremental study must be approved for
cross-course use. State actual individual contributions and AI assistance before
submitting. See `proposal.md`, `course_contributions.md` and `AI_USAGE.md`.

### Source-derived course requirements versus proposed research
| Source supplied by the student | Requirement supported by that source | This proposed contribution |
|---|---|---|
| AMS 518, Fall 2026 syllabus p.2 | Each student completes/presents a numerical case study; peers' projects reviewed | Constrained CVaR hedging case study, later stage |
| AMS 518, p.3 | AI forbidden for homework; project AI use clearly acknowledged | Project-only AI disclosure and human validation log |
| AMS 520, pp.2–3 | Interactive Colab/Jupyter report, three presentations, GitHub code/docs without data credentials | This notebook is a Phase I starting point, not a substitute for actual data collection |
| AMS 520, p.3 | Phase I Lecture 10: literature, data collection, cleaning; Phase II Lecture 22: EDA/first empirical results; Phase III final-exam day | Sequence the data/learning/optimization work against the confirmed course schedule |

The syllabi do **not** establish shared-submission approval or calendar dates here.
The AMS 520 AI policy and detailed AMS 518 project requirements still need clarification.
Private syllabus PDFs, meeting links and passcodes are not redistributed.

## 1. Research question and literature map
**Question:** Does order-book information improve the out-of-sample cost–risk tradeoff
of derivative hedging, and is the conclusion robust to volatility dynamics?

| Reference | Relevant result/framework | Use and limitation in this project |
|---|---|---|
| [Cont, Kukanov & Stoikov, *The Price Impact of Order Book Events*](https://arxiv.org/abs/1011.6402) | Empirical relationship between order-flow imbalance, depth and price changes | Motivate book features; contemporaneous impact is not a forecasting result for our data |
| [Rockafellar & Uryasev, *Optimization of Conditional Value-at-Risk* (1999 manuscript)](https://sites.math.washington.edu/~rtr/papers/rtr179-CVaR1.pdf) | Scenario-based CVaR optimization | Later constrained hedge decision, not implemented in Phase I |
| [Jourdain & Zhou, *Existence of a calibrated regime switching local volatility model and new fake Brownian motions*](https://arxiv.org/abs/1607.00077) | LSV calibration/existence issues involving conditional variance | Distinguish chosen leverage functions from calibrated LSV; controlled dynamics later |
| [Bühler et al., *Deep Hedging*](https://arxiv.org/abs/1802.03042) | Learning hedges with frictions and convex risk objectives | Optional sequential-learning extension, not the first baseline |
| [IBKR Market Depth introduction](https://www.interactivebrokers.com/docs/tws-api/doc/market-data-live/market-depth-l-2/introduction) and [request interface](https://www.interactivebrokers.com/docs/tws-api/doc/market-data-live/market-depth-l-2/request-market-depth) | Native depth rows, direct/Smart selection, incomplete quoted-price coverage, odd-lot exclusion and reset handling | Defines what our feed can and cannot establish |

These are external research sources. They are not additional syllabus requirements.
See `references.bib` for bibliographic metadata. None implies a completed empirical result.

## 2. Select input and record provenance
Run from this repository or set `DTS_REPO_ROOT` to its absolute path. To analyze a
real stopped-session export, set `DTS_DEPTH_EXPORT` to its file before starting a
fresh kernel. Never put a dashboard/IBKR credential in notebook cells. A supplied
but invalid path raises an error; it does not silently switch to synthetic data.

The export hash detects content changes, not malicious provenance forgery. A session
is local acquisition metadata, not proof that every exchange event was observed.

In [ ]:
import os
from pathlib import Path
import sys
import hashlib
import json
import importlib.metadata

root_hint = os.environ.get("DTS_REPO_ROOT", "")
candidates = [Path(root_hint).expanduser()] if root_hint else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p.resolve() for p in candidates
             if (p / "research/liquidity_aware_hedging/depth_replay.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set DTS_REPO_ROOT to the checkout containing the research module.")
MODULE = ROOT / "research/liquidity_aware_hedging/depth_replay.py"
sys.path.insert(0, str(MODULE.parent))
from depth_replay import Book, load_export, synthetic_fixture, replay, validate_export
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

export_path = os.environ.get("DTS_DEPTH_EXPORT", "").strip()
payload = load_export(export_path) if export_path else synthetic_fixture()
validate_export(payload)
SYNTHETIC = payload["session"]["source"] == "mock"
LABEL = "SYNTHETIC protocol fixture" if SYNTHETIC else "IBKR delivered displayed-depth session"
print(LABEL)
print("Not a complete exchange book, fill record or forecasting result.")
provenance = {
    "source": payload["session"]["source"], "symbol": payload["session"]["symbol"],
    "venue": payload["session"]["venue"], "rows_requested": payload["session"]["requested_rows"],
    "terminal_state": payload["session"]["state"], "events": len(payload["events"]),
    "export_sha256": payload["sha256"],
    "python_replay_sha256": hashlib.sha256(MODULE.read_bytes()).hexdigest(),
    "versions": {p: importlib.metadata.version(p) for p in ("numpy", "pandas", "matplotlib")},
}
display(provenance)

## 3. Deterministic reconstruction before statistics
Operations are applied in the **local subscription sequence**: insert 0, update 1,
delete 2; ask side 0, bid side 1. Row positions shift; never treat them as permanent
price identifiers. Maker rows at the same price are aggregated for top-price features.

A reset clears both sides. A local gap or impossible position invalidates the book;
no interpolation or forward fill restores a missing state. The label
`two_sided_unverified` means structural checks passed, not complete/fresh/tradable.
Crossed, locked and zero-size states stay in the quality report and are excluded
from displayed-cost diagnostics. A halted/interrupted subscription is not an empty
tradable book. Quality classification and all excluded segments must remain visible.

In [ ]:
states = list(replay(payload))
assert states == list(replay(payload)), "Replay must be deterministic."
assert len(states) == len(payload["events"])
# Preserve the original strings; derived numeric diagnostics never replace raw fields.
assert all(isinstance(e["size"], str) for e in payload["events"])
for row in states:
    if row["kind"] == "reset":
        assert row["bids"] == [] and row["asks"] == [] and row["spread"] is None

df = pd.DataFrame(states)
df["sequence_number"] = df["sequence"].map(int)
df["receipt_utc"] = pd.to_datetime(df["received_unix_us"].map(int), unit="us", utc=True)
quality = df["quality"].value_counts(dropna=False).rename_axis("quality").to_frame("event_count")
quality["event_fraction"] = quality["event_count"] / len(df) if len(df) else np.nan
display(quality)
display(df[["sequence", "kind", "origin", "epoch", "quality", "receipt_utc"]].head(15))

## 4. Timestamp and boundary diagnostics
Unix timestamps are local callback-handling receipt times, not exchange times.
Monotonic timestamps support within-process elapsed-time checks, not cross-session
alignment. Recovery markers have no monotonic receipt (`0`). A quiet interval may
mean no delivered updates, slow handling or an outage: elapsed time alone cannot
establish which. Event frequencies/means below are **event-weighted**, not trading
volume-weighted or time-weighted.

No 30-second prediction labels are built yet. That later stage must define a causal
clock-grid sampler, age limits, resets, target availability and split purging.

In [ ]:
boundary_mask = df["kind"].isin(["start", "reset", "stop", "gap", "error", "interrupted"])
display(df.loc[boundary_mask, ["sequence", "kind", "origin", "epoch", "quality", "receipt_utc"]])
local_seq = df["sequence"].map(int)
mono = df["received_monotonic_ns"].map(int)
valid_time_pair = mono.gt(0) & mono.shift(1).gt(0) & df["epoch"].eq(df["epoch"].shift(1))
interarrival_ms = mono.diff().where(valid_time_pair) / 1_000_000
report = {
    "local_sequence_discontinuities": int(local_seq.diff().dropna().ne(1).sum()),
    "backward_local_monotonic_observations": int(interarrival_ms.lt(0).sum()),
    "reset_markers": int(df["kind"].eq("reset").sum()),
    "interruption_or_error_markers": int(df["kind"].isin(["gap", "error", "interrupted"]).sum()),
    "eligible_two_sided_states": int(df["quality"].eq("two_sided_unverified").sum()),
    "unknown_upstream_loss_count": True,
}
display(report)
display(interarrival_ms.dropna().describe().to_frame("local_interarrival_ms"))

## 5. Spread and depth diagnostics
For best observed direct bid $b$ and ask $a$, the displayed spread is $a-b$ and
midpoint $(a+b)/2$. Total observed row quantities give
$I=(Q_b-Q_a)/(Q_b+Q_a)$. This is **depth imbalance**, not order-flow imbalance.
The figures use local event order. Gaps remain gaps; reset epochs are plotted
separately so a line never connects two reconstruction sessions.

Rows and sizes are the provider's reported quantities. Do not multiply by 100 or
infer a universal lot convention. Confirm units for the actual instrument/feed
before economic interpretation.

In [ ]:
eligible = df["quality"].eq("two_sided_unverified")
summary = df.loc[eligible, ["spread", "bid_depth", "ask_depth", "depth_imbalance", "top_imbalance"]].describe()
display(summary)
fig, ax = plt.subplots(figsize=(9, 3.5))
for epoch, part in df.groupby("epoch", sort=False):
    ax.plot(part["sequence_number"], part["spread"], label=f"Epoch {epoch}")
ax.set(xlabel="Local event sequence", ylabel="Observed spread (price units)", title=LABEL + " — direct displayed spread")
ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for epoch, part in df.groupby("epoch", sort=False):
    ax.plot(part["sequence_number"], part["depth_imbalance"], label=f"Epoch {epoch}")
ax.set(xlabel="Local event sequence", ylabel="Observed depth imbalance", ylim=(-1.05, 1.05), title=LABEL + " — row-depth imbalance")
ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for column in ("bid_depth", "ask_depth"):
    # Keep invalid-state NaNs, including reset rows, in each line.
    ax.plot(df["sequence_number"], df[column], label=column)
ax.set(xlabel="Local event sequence", ylabel="Reported quantity units", title=LABEL + " — recorded row quantities")
ax.legend(); fig.tight_layout(); plt.show()

## 6. A displayed crossing-cost proxy — not a fill simulator
For a hypothetical buy size $q>0$, walk the observed asks until $q$ is covered.
Subtract $q$ times the current midpoint from the displayed notional. Reverse the
sign for a sale. A quantity beyond recorded depth yields **unavailable**, not an
extrapolated execution. This calculation omits latency, hidden liquidity, odd lots,
fees, impact and other traders. It does not prove a trade could have executed.

The sizes below are an editable protocol illustration. Before an empirical study,
choose economically relevant sizes from pilot/training data and predeclare them.
No cost prediction or hedge decision is made here.

In [ ]:
illustrative_sizes = [1.0, 10.0, 100.0, 1_000_000.0]
book = Book(payload["session"]["requested_rows"])
cost_rows = []
chosen_sequence = None
for event in payload["events"]:
    book.apply(event)
    if book.quality() == "two_sided_unverified" and len(book.asks) == book.rows and len(book.bids) == book.rows:
        chosen_sequence = event["sequence"]
        for q in illustrative_sizes:
            cost_rows.append({"quantity": q, "buy_cost_proxy": book.displayed_cost(q), "sell_cost_proxy": book.displayed_cost(-q)})
        break
print("Illustrative local sequence:", chosen_sequence)
if not cost_rows:
    print("No structurally usable fully populated observed state; no cost illustration created.")
else:
    display(pd.DataFrame(cost_rows))
    print("NaN means unavailable beyond the observed book, not zero cost.")

## 7. Cleaning decisions and proposed temporal validation
Keep raw events immutable. Exclude invalid reconstructed states from learning, but
report exclusion counts/reasons. Do not infer omitted observations or fill across
resets. Separate multiple venues and sessions. Build a documented age-aware sampler
before attaching 30-second outcomes. Do not use an entire-session quality summary
as a feature in an earlier historical decision.

**Planned AMS 520 tests:** persistence versus regularized regression versus one
nonlinear model; train on earlier sessions, validate/test on later sessions; purge
labels overlapping splits; fit scalers and thresholds on training only; report
per-session performance and price-only/top-book/depth ablations. None has been run
on empirical data in this notebook.

**Planned AMS 518 tests:** periodic delta versus constrained CVaR without book
information versus the same optimizer with book information. Keep liability,
initial funding, scenario paths and constraints fixed. Predicted future liquidity
enters future scenario costs, not the currently observed execution-cost proxy.
Convexity requires appropriate convex costs/exogenous scenarios; fixed trade fees
need separate treatment. Controlled LSV dynamics are a later robustness experiment,
not a calibrated joint microstructure model.

## 8. Phase I completion record — fill with actual work

| Item | Current status / student evidence required |
|---|---|
| Instructor approval for shared infrastructure | Not yet supplied |
| AMS 520 AI-use conditions | Clarification required |
| Group membership and individual contribution | Not supplied; do not invent members/contributions |
| Verified real depth access and units | Not established by this synthetic run |
| Collection dates, venue, session lengths, outages | Fill after actual pilot |
| Redistribution/storage conditions | Confirm with provider before sharing data/outputs |
| Cleaning/replay acceptance on real session | Record exact export hash and tests run |
| Literature survey | Starting map provided; add critical reading notes and source-supported conclusions |
| Human validation and understanding | Document code review, derivations, tests and limitations personally verified |

**AI disclosure:** ChatGPT assisted substantially with design, code, tests, notebook
and prose drafting; assistance was not limited to editing. The student is responsible
for checking and explaining submitted work. This is project work, not AMS 518 homework.
A runnable synthetic fixture is software evidence, not evidence that Phase I's real
data collection and academic requirements have been completed.

In [ ]:
print("Completed notebook mode:", LABEL)
print("Reconstruction events:", len(df))
print("Export fingerprint:", payload["sha256"])
print("No predictor, CVaR optimizer, LSV calibration, or real-broker acceptance is claimed.")